# 🧹 Meningioma Cleaning Notebook

Builds the analysis-ready cohort and modelling datasets.

Run from the **repo root** (`meningioma-atypier/`). Run top to bottom. Hands off to `meningioma-modelling.ipynb` via `output/datasets/`.


Run top to bottom. Creates `output/datasets/unimputed_df.parquet` plus a MICE or simple modelling parquet; §12 validates every saved parquet before handoff.

<details>
<summary><b>Pipeline map</b> — notebook step → <code>config/*.py</code></summary>

Each step calls a config module (the pipeline engine) or a shared helper. Study-specific choices live in the notebook cells; the `config/` files are just the machinery.

| Step | What it does | Driven by |
|------|--------------|-----------|
| 00 | Setup — imports, reset `output/` | — |
| 01 | Load raw export | `config/cohort.py · load_raw` |
| 02 | Rename columns → snake_case | `config/column_rename_map.py` |
| 03 | Schema — declare + coerce | `config/schema_overrides.py` · `cleaning.apply_schema` |
| 04 | Duplicate audit | `cleaning.audit_duplicates` |
| 05 | Row filters (year cohort + inclusion) | `config/cohort.py` · `config/row_filters.py` |
| 06 | Missingness story (diagnostic) | `missingness_resolution.analyze_missingness` |
| 07 | Missingness policy (structural / MNAR) | `config/missingness.py` |
| 08 | Derivations (bins, flags, computed cols) | `config/derivations.py` |
| 09 | Schema validation (pandera) | `pandera` |
| 10 | DDA + pre-imputation peek | `dda.run_dda` / `run_dda_bivariate` / `run_dda_trivariate` |
| 11 | Imputation (MICE or simple) | `missingness_resolution` |
| 12 | Save + validate handoff datasets | `dataset_handoff` |

`config/analysis.py` and `config/report_settings.py` are used by `meningioma-modelling.ipynb`, not here.

</details>


## 00 · Setup

⚙️ Boots the pipeline and gives you a clean slate.

<details>
<summary>🔧 How it works</summary>

- 📦 `from heavy_machinery.cleaning_phase…` and `from heavy_machinery.config import load`.
- 🧹 Wipes and recreates `output/` so every run starts from scratch.
- 📅 Cohort year scope (`ANALYSIS_YEARS`) is set in the year-filter cell under §05.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Load a config module by number**
```python
_cohort = load("cohort")          # returns the module object
```

**Cohort year scope** (set in §05 year-filter cell)
```python
ANALYSIS_YEARS = None             # 🌍 all years
ANALYSIS_YEARS = [2025]           # 📌 one cohort year
ANALYSIS_YEARS = [2024, 2025]     # 🗓️ a subset
```

</details>


In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import pandera.pandas as pa

import numpy as np
import shutil
from pathlib import Path
from collections.abc import Sequence

from IPython.display import display

from heavy_machinery.config import load
from heavy_machinery.cleaning_phase.schema_infer import (
    infer_schema, print_schema_template, print_column_uniques, schema_summary, ColSpec,
)
from heavy_machinery.cleaning_phase.cleaning import apply_schema, audit_duplicates
from heavy_machinery.cleaning_phase.dda import (
    run_dda, run_dda_bivariate, run_dda_trivariate, build_dda_bivariate_specs,
)
from heavy_machinery.cleaning_phase.missingness_resolution import (
    analyze_missingness, proper_mice_impute, rf_chained_impute, simple_impute_stage,
)
from heavy_machinery.cleaning_phase.validation import category_validation, pandera_template, pandera_validate
from heavy_machinery.cleaning_phase.dataset_handoff import validate_handoff_datasets

OUTPUT_ROOT = Path("output")
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)



## 01 · Load raw data

📥 Reads the raw export into a dataframe — nothing else yet.

<details>
<summary>🔧 How it works · <code>config/cohort.py · load_raw</code></summary>

- 📄 `load_raw(DATA_PATHS)` reads one or more exports — **CSV or Excel**, picked automatically by file extension.
- 🔗 Pass a single path string or a list of paths; multiple files are stacked with `pd.concat`.
- 🔢 Prints the `rows × columns` count per file (and combined total when stacking).
- 🚫 No filtering or type coercion happens here.
- 👀 `df.head(0)` shows just the raw column headers, which feed the rename map in §02.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Single CSV export**
```python
DATA_PATHS = ["heavy_machinery/Meningiomas PSKUS grants - Visi pacienti.csv"]
df_raw = _cohort.load_raw(DATA_PATHS)
```

**Single Excel export** — extension decides the reader
```python
DATA_PATHS = ["cohort_export.xlsx"]
df_raw = _cohort.load_raw(DATA_PATHS)
```

**Multiple exports** — mix CSV and Excel; same columns expected
```python
DATA_PATHS = [
    "cohort_2024.csv",
    "cohort_2025.xlsx",
]
df_raw = _cohort.load_raw(DATA_PATHS)
```

</details>


In [ ]:
DATA_PATHS = [
    "Meningiomas PSKUS grants - Visi pacienti.csv",
]
# Stack more exports here, e.g.:
# DATA_PATHS = ["cohort_2024.csv", "cohort_2025.xlsx"]

_cohort = load("cohort")
df_raw = _cohort.load_raw(DATA_PATHS)
#df_raw.head(0)

## 02 · Rename columns

🏷️ Turn messy raw headers into clean `snake_case` names.

1. ▶️ Run **see raw columns** → copy the printed skeleton
2. ✍️ Paste into `COLUMN_RENAME_MAP` and fill in the snake_case names
3. ✅ Run **apply rename**

<details>
<summary>🔧 How it works · <code>config/column_rename_map.py</code></summary>

- 🖨️ `list_cols(df_raw)` prints a copy-paste `COLUMN_RENAME_MAP` skeleton — one row per raw column.
- ✍️ You fill in the right-hand snake_case names.
- 🔁 `apply_rename(df_raw, COLUMN_RENAME_MAP)` returns the renamed frame.
- ⏱️ Renaming runs **before** schema inference, so every later step uses the clean names.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Step 1 — print the skeleton**
```python
load("column_rename_map").list_cols(df_raw)
```

**Step 2 — fill it in**
```python
COLUMN_RENAME_MAP = {
    "Vecums. gadi": "age",
    "WHO pakāpe (2021). 1 / 2 / 3": "who_grade",
    # ... one entry per raw column
}
```

**Step 3 — apply**
```python
df_raw = load("column_rename_map").apply_rename(df_raw, COLUMN_RENAME_MAP)
df = df_raw
```

</details>


In [ ]:
#🟧🟧🟧 Step 1 — see raw columns (run once per new dataset)

#load("column_rename_map").list_cols(df_raw)

In [ ]:
#🟧🟧🟧 Step 2 — paste skeleton here and fill in the right-hand names

COLUMN_RENAME_MAP = {
    "Nr.": "id",
    "Personas kods": "patient_code",
    "Unnamed: 2": "entry_year",
    "Vecums. gadi": "age",
    "Dzimums. 0 - vīrietis\n1 - sieviete\"": "sex",
    "Histoloģija. 0 - nav\n1 - ir": "histology_available",
    "WHO pakāpe (2021). 1 / 2 / 3": "who_grade",
    "Progesterons. 0 - negatīvs\n1 - pozitīvs": "progesterone_pos",
    "Ki-67 (%). skaitlis. %": "ki67_pct",
    "Smadzeņu parenhīmas invāzija. 0 - nav\n1 - ir": "brain_invasion",

    "Nekroze histoloģiski. 0 - nav\n1 - ir": "hist_necrosis",
    "MRI izmeklējuma datums": "mri_date",
    "Puse. 1 - labā\n2 - kreisā\n3 - viduslīnija": "side",
    "Lokalizācija: skull base / non–skull base. 0 - non-skull base\n1 - skull base": "tumor_location",
    "Cik meningiomas?": "meningioma_count",

    "Max diametrs. skaitlis.cm": "max_diameter_cm",
    "Tilpums": "tumor_volume",
    "Pamatmodalitāte analīzei. 0 - MRI\n1 - CT\n3 - MRI+CT": "additional_ct",
    "K/v i/v. 0 - nav\n1 - ir": "iv_contrast",
    "0 - primārs\n1 - recidīvs": "tumor_episode",
    "Audzēja robeža. 1 = gluda. \n2 = neregulāra": "tumor_margin",
    "Dural tail sign. 0 - nav\n1 - ir": "dural_tail",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement). 0 - nav\n1 - ir": "capsular_enhancement",
    "Kontrastēšanās veids. 0 - homogēna\n1 - heterogēna": "heterogeneous_enhancement",
    "Perifokāla tūska. 0 - nav\n1 - ir": "perifocal_edema",

    "Perifokālas tūskas tilpums. cm3": "edema_volume_cm3",
    "Masas efekts. 0 - nav\n1 - ir": "mass_effect",
    "Audzēja kalcifikācija. 0 - nav\n1 - ir": "calcification",
    "Cistiskas komponentes. 0 - nav\n1 - ir": "cystic_component",
    "Audzēja nekroze. 0 - nav\n1 - ir": "mri_necrosis",
    "Hemorāģiskas sastāvdaļas. 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams ": "hemorrhage",
    "Kaule hiperostoze. 0 - nav\n1 - ir": "hyperostosis",
    "Kaula invāzija (cortical destruction). 0 - nav\n1 - ir": "cortical_destruction",
    "Tumor Hyperintensity on DWI. 0 - nav\n1 - ir": "dwi_hyperintensity",
    "Tumor Hyperintensity on T2. 0 - nav\n1 - ir": "t2_hyperintensity",
    "Tumor Hypointensity on T1. 0 - nav\n1 - ir": "t1_hypointensity",

    "Sīnuss. 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug": "sinus_invasion",
    "Cauraug falx cerebri 0 - nav. 1 - ir": "transfalcine_extension",
    "ADC map value": "adc_value",
    }

In [ ]:
#🟧🟧🟧 Step 3 — apply rename

df_raw = load("column_rename_map").apply_rename(df_raw, COLUMN_RENAME_MAP)
df = df_raw
#df.head(0)

## 03 · Schema

🧱 Declare each column's `ColSpec`, then coerce the frame to those types.

1. 🔎 Run **infer**
2. 🖨️ Run **print template**
3. 🧪 Run **column uniques** (nulls / replace hints)
4. ✍️ Edit **schema_overrides**
5. ✅ Run **apply overrides**
6. 🔧 Run **apply_schema** (coerce dtypes)

<details>
<summary>🔧 How it works · declare · <code>config/schema_overrides.py</code></summary>

- 🤖 `infer_schema` guesses a `ColSpec` (kind + levels) per column.
- 🖨️ `print_schema_template` prints those guesses as an editable `schema_overrides` dict.
- ✍️ You correct the `kind`, `replace` maps, `nulls`, and `keep` flags.
- 🔗 `apply_schema_overrides(...)` merges your edits and writes `schema/schema_summary.csv`.

</details>

<details>
<summary>🔧 How it works · coerce · <code>cleaning_phase/cleaning.py · apply_schema</code></summary>

- 🔁 Applies each column's `replace` map.
- 🧮 Casts to the right dtype — datetime, ordered categorical, numeric, etc.
- 🚫 Maps declared `nulls` to `NaN`.
- 🧾 Writes `output/cleaning/schema_coercion.csv` — which values became what (incl. `(missing)`), with counts.
- 🗑️ Drops columns flagged `keep=False`.
- 📝 `schema_log` records every action for the report.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Infer + print the editable template**
```python
schema = infer_schema(df_raw)
print_schema_template(schema)
```

**The `ColSpec` kinds you can declare**
```python
ColSpec(name="id",               kind="id")
ColSpec(name="age",              kind="continuous")
ColSpec(name="brain_invasion",   kind="binary")
ColSpec(name="sex",              kind="nominal",  replace={0: "male", 1: "female"})
ColSpec(name="who_grade",        kind="ordinal",  ordered_levels=["1", "2", "3"])
ColSpec(name="mri_date",         kind="datetime", datetime_bin="full", keep=False)
ColSpec(name="meningioma_count", kind="count")
ColSpec(name="progesterone_pos", kind="binary",   nulls=(2,))    # 🚫 map 2 → NaN
ColSpec(name="patient_code",     kind="id",        keep=False)   # 🗑️ drop after cleaning
```

**Merge overrides + coerce**
```python
load("schema_overrides").apply_schema_overrides(schema, schema_overrides, OUTPUT_ROOT)
schema_log = []
df = apply_schema(df, schema, log=schema_log, output_root=OUTPUT_ROOT)
n_rows_after_schema = len(df)
```

</details>


In [ ]:
schema = infer_schema(df_raw)
#schema_summary(schema)

In [ ]:
#print_schema_template(schema)

In [ ]:
#🟧🟧🟧 inspect raw values — use for nulls=() and replace={} below
#print_column_uniques(df, schema)

In [ ]:
#🟧🟧🟧 Edit overrides, then run
schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind="id"),
    
    'entry_year': ColSpec(name='entry_year', kind='count'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='nominal', replace={0:"male", 1:"female",}),
    
    'who_grade': ColSpec(name='who_grade', kind='ordinal', ordered_levels=["1","2","3"]),
    
    #'histology_available': ColSpec(name='histology_available', kind='binary', nulls=(2,)),
    'histology_available': ColSpec(name='histology_available', kind='skip'),

    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary', nulls=(2,)),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    
    'mri_date': ColSpec(name='mri_date', kind='datetime', datetime_bin='full'),
    
    'side': ColSpec(name='side', kind='nominal', replace={'1': "right", '2': "left", '3': "midline"}),
    'tumor_location': ColSpec(name='tumor_location', kind='nominal', replace={0: "non_skull_base", 1: "skull_base"}, nulls=(2,)),
    'meningioma_count': ColSpec(name='meningioma_count', kind='count'),
    'max_diameter_cm': ColSpec(name='max_diameter_cm', kind='continuous'),
    'tumor_volume': ColSpec(name='tumor_volume', kind='continuous'),

    'additional_ct': ColSpec(name='additional_ct', kind='skip'),
    'iv_contrast': ColSpec(name='iv_contrast', kind='skip'),    
    #'additional_ct': ColSpec(name='additional_ct', kind='binary', replace={0.0: False, 1.0: pd.NA, 3.0: True}),
    #'iv_contrast': ColSpec(name='iv_contrast', kind='binary'),

    'tumor_episode': ColSpec(name='tumor_episode', kind='ordinal', replace={'0': "primary", '1': "recurrent"}, ordered_levels=["primary", "recurrent"], nulls=("multiplas",)),
    
    'tumor_margin': ColSpec(name='tumor_margin', kind='nominal', replace={1: "regular", 2: "irregular"}, nulls=(0,)),
    'dural_tail': ColSpec(name='dural_tail', kind='binary'),
    
    'capsular_enhancement': ColSpec(name='capsular_enhancement', kind='binary'),
    'heterogeneous_enhancement': ColSpec(name='heterogeneous_enhancement', kind='binary'),
    'dwi_hyperintensity': ColSpec(name='dwi_hyperintensity', kind='binary', nulls=('-',)),
    't2_hyperintensity': ColSpec(name='t2_hyperintensity', kind='binary', nulls=('-',)),
    't1_hypointensity': ColSpec(name='t1_hypointensity', kind='binary', nulls=('-',)),
    
    'perifocal_edema': ColSpec(name='perifocal_edema', kind='binary'),
    'edema_volume_cm3': ColSpec(name='edema_volume_cm3', kind='continuous'),
    
    'mass_effect': ColSpec(name='mass_effect', kind='binary'),
    'calcification': ColSpec(name='calcification', kind='binary'),
    'cystic_component': ColSpec(name='cystic_component', kind='binary'),
    'mri_necrosis': ColSpec(name='mri_necrosis', kind='binary'),
    'hemorrhage': ColSpec(name='hemorrhage', kind='binary', nulls=(2.0,)),
    'hyperostosis': ColSpec(name='hyperostosis', kind='binary'),
    'sinus_invasion': ColSpec(name='sinus_invasion', kind='ordinal', replace={0: "no_invasion", 1: "sinus_invasion", 2: "transsinus_extension"}, ordered_levels=["no_invasion", "sinus_invasion", "transsinus_extension"]),
    'cortical_destruction': ColSpec(name='cortical_destruction', kind='binary'),
    'transfalcine_extension': ColSpec(name='transfalcine_extension', kind='binary'),
    
    'adc_value': ColSpec(name='adc_value', kind='continuous'),
    }

load("schema_overrides").apply_schema_overrides(schema, schema_overrides, OUTPUT_ROOT)

In [ ]:
schema_log = []
df = apply_schema(df, schema, log=schema_log, output_root=OUTPUT_ROOT)
n_rows_after_schema = len(df)
#df.head()
#pd.read_csv(OUTPUT_ROOT / "cleaning" / "schema_coercion.csv")


## 04 · Duplicate audit

🕵️ Find rows that share the same identifier(s).

<details>
<summary>🔧 How it works · <code>cleaning_phase/cleaning.py · audit_duplicates</code></summary>

- 👥 Groups rows sharing the same `id_cols` and returns the duplicate groups for review.
- 👀 `drop=False` → report only (default, safe).
- ✂️ `drop=True` → actually remove duplicates (use only once confirmed real).
- 1️⃣ `include_first=True` keeps the first occurrence visible in the report.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Report only (recommended first pass)**
```python
dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
dupes.head() if len(dupes) else print("No duplicate groups found.")
```

**Drop confirmed duplicates**
```python
dupes, df = audit_duplicates(df, id_cols=ID_COLS, drop=True)
```

</details>


In [ ]:
ID_COLS = ["id", "patient_code", "entry_year"]
dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
#dupes.head() if len(dupes) else print("No duplicate groups found.")

## 05 · Row filters

🧹 Trim the cohort **after** typing. Year scope is its **own** cell; inclusion rules are the next cell; then finalize.

<details>
<summary>🔧 How it works</summary>

- 📅 **Year filter** (`config/cohort.py · apply_cohort_year_filter`) — optional `ANALYSIS_YEARS` subset. `None` = all years. Logged into drop_log.
- ✅ **Inclusion filters** (`config/row_filters.py`) — `who_grade` exists, MRI exists, etc. Toggle `active`.
- 💾 `finalize_row_drops` writes cleaning artifacts for the report.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Year filter (separate cell)**
```python
YEAR_COLUMN = "entry_year"
ANALYSIS_YEARS = None              # 🌍 all years
df, cohort_year_log = _cohort.apply_cohort_year_filter(df, YEAR_COLUMN, ANALYSIS_YEARS)
```

**Inclusion filters** (prepend year log → one table for peek + drop_log)
```python
df, post_schema_row_filter_log = _row_filters.apply_row_filters(df, post_schema_row_filters)
post_schema_row_filter_log = pd.concat(
    [cohort_year_log, post_schema_row_filter_log], ignore_index=True,
)
post_schema_row_filter_log
```

**Finalize**
```python
df = _row_filters.finalize_row_drops(
    df, post_schema_row_filter_log, output_root=OUTPUT_ROOT, df_raw=df_raw,
    n_rows_after_schema=n_rows_after_schema,
    schema=schema, dupes=dupes, schema_log=schema_log,
)
```

</details>


In [ ]:
#🟧🟧🟧 Cohort year filter (separate from inclusion rules below)
YEAR_COLUMN = "entry_year"
ANALYSIS_YEARS: list[int] | None = None   # e.g. [2025]; None = all years
#ANALYSIS_YEARS = [2025]

df, cohort_year_log = _cohort.apply_cohort_year_filter(df, YEAR_COLUMN, ANALYSIS_YEARS)
#cohort_year_log

In [ ]:
_row_filters = load("row_filters")

post_schema_row_filters = [
    _row_filters.RowFilter(
        name="Meningioma location - brain",
        keep=lambda d: d["side"].astype("string").isin(["right", "left", "midline"]),
        note="Intracranial meningioma: side recorded as right, left or midline",
        active=True,
    ),
    _row_filters.RowFilter(
        name="who_grade exists",
        keep=lambda d: d["who_grade"].astype("string").isin(["1", "2", "3"]),
        note="Histologically confirmed WHO grade 1, 2 or 3 recorded",
        active=True,
    ),
    _row_filters.RowFilter(
        name="MRI is one of the modalities",
        keep=lambda d: d["mri_date"].notna(),
        note="Preoperative MRI performed (mri_date present)",
        active=True,
    ),
]

df, post_schema_row_filter_log = _row_filters.apply_row_filters(df, post_schema_row_filters)
post_schema_row_filter_log = pd.concat(
    [cohort_year_log, post_schema_row_filter_log],
    ignore_index=True,
)
#post_schema_row_filter_log

In [ ]:
df = _row_filters.finalize_row_drops(
    df, post_schema_row_filter_log,
    output_root=OUTPUT_ROOT,
    df_raw=df_raw,
    n_rows_after_schema=n_rows_after_schema,
    schema=schema,
    dupes=dupes,
    schema_log=schema_log,
)

## 06 · Missingness story

🕳️ Quantify what's missing, before deciding what to do about it.

<details>
<summary>🔧 How it works · <code>cleaning_phase/missingness_resolution.py · analyze_missingness</code></summary>

- 📉 `analyze_missingness(df, output_root=OUTPUT_ROOT)` reports missingness per column (count + %).
- 🖼️ Saves the missingness tables/figures under `output/missingness/`.
- 🔍 Purely diagnostic — tells you *which* columns are missing and *how much*.
- ➡️ You decide the actual policy in §07.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Summarise + show only the columns with gaps**
```python
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary[missing_summary.n_missing > 0]
```

</details>


In [ ]:
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary[missing_summary.n_missing > 0]

## 07 · Missingness policy

🧭 Tell the pipeline *how* to treat missing values — declare it, then apply once.

<details>
<summary>🔧 How it works · <code>config/missingness.py</code></summary>

Two declarative knobs decide what gets imputed in §11:

- 🧩 **`StructuralGroup`** — slot-style columns where NaN means *the slot does not exist*. Instead of imputing, it derives a count/max and marks the raw columns `skip`.
- 🚩 **`MnarColumn`** — missing-not-at-random: missingness itself may be informative, so it adds a binary `<col>_missing` flag to the schema.
- ▶️ `apply_missingness_policy` applies both and returns updated `df`, `schema`, and an audit log.
- 🪹 Both lists are empty by default — add entries only when a column genuinely fits.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Structural group — derive count/max, skip the raw slots**
```python
_missingness = load("missingness")

STRUCTURAL_GROUPS = [
    _missingness.StructuralGroup(
        name="lesion_mri_pirads",
        cols=["lesion_1_MRI_PIRADS", "lesion_2_MRI_PIRADS", "lesion_3_MRI_PIRADS"],
        derive_count_col="n_mri_pirads_lesions",
        derive_max_col="max_mri_pirads",
        skip_raw=True,
        reason="Blank lesion slots mean the lesion does not exist, not unknown.",
    ),
]
```

**MNAR column — add an informative `<col>_missing` flag**
```python
MNAR_COLUMNS = [
    _missingness.MnarColumn(
        col="ki67_pct", flag_col="ki67_pct_missing",
        reason="Ki-67 may be absent because it was not measured/reported.",
    ),
]
```

**Apply both (empty lists are fine — nothing happens)**
```python
df, schema, missingness_log = _missingness.apply_missingness_policy(
    df=df, schema=schema,
    structural_groups=STRUCTURAL_GROUPS,
    mnar_columns=MNAR_COLUMNS,
)
```

</details>


In [ ]:
_missingness = load("missingness")

STRUCTURAL_GROUPS = []

MNAR_COLUMNS = []

In [ ]:
df, schema, missingness_log = _missingness.apply_missingness_policy(
    df=df,
    schema=schema,
    structural_groups=STRUCTURAL_GROUPS,
    mnar_columns=MNAR_COLUMNS,
    )

#missingness_log

## 08 · Derivations (pre-imputation)


🧬 Build new analysis columns from the cleaned ones — declare them in `DERIVATIONS` (no `.py` edits needed).

<details>
<summary>🔧 How it works · <code>config/derivations.py</code></summary>

All study-specific logic lives in the notebook; `config/derivations.py` is just the engine. Three building blocks:

- 📊 **`BinNumeric`** — cut a numeric column into ordered bins (`age` → `age_bins`). Bins = edge values, labels = one per gap (`len(bins) - 1 == len(labels)`). Default `right=False` gives left-closed intervals (`[50, 60)` → `"50-59"`).
- 🔧 **`Apply`** — single-column custom logic via `fn=lambda s: ...` (Ki-67 midpoint, grouped labels, boolean flags).
- 🧮 **`Compute`** — needs several columns; `fn` receives the whole frame (e.g. zeroing `edema_volume_cm3` when there is no perifocal edema).

Per-entry switches:

- 📝 `rule=` is the definition ("`tumor_volume` ≥ 13.95 cm³"); `reason=` is the citation it came from. Both are exported to `derivation_log.csv` and shown as separate columns in the report.
- 🔀 `active=False` skips an entry.
- ♻️ `overwrite=True` replaces an existing column.
- ▶️ `apply_derivations` runs the list, updates the schema, previews new columns, and (with `write_csv=True`) writes the cleaned dataset + derivation log.
- 🔁 `DERIVED_DEPENDENCIES` + `derivations=DERIVATIONS` re-apply the same rules after R MICE in §11.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**`BinNumeric` — ordered age bands**
```python
_derivations = load("derivations")

_derivations.BinNumeric(
    name="age_bins", source="age",
    bins=[-np.inf, 50, 60, 70, 80, np.inf],
    labels=["<50", "50-59", "60-69", "70-79", "80+"],
    kind="ordinal", reason="Age groups for descriptive tables.",
)
```

**`Apply` — boolean flag and an ordinal group**
```python
_derivations.Apply(name="high_grade", source="who_grade",
           fn=lambda s: s.astype("Float64").pipe(lambda sf: (sf == 2) | (sf == 3)),
           kind="binary", reason="WHO grade 2/3 = high-grade.")

_derivations.Apply(name="ki67_group", source="ki67_mid", fn=lambda s: s.map(_ki67_group),
           kind="ordinal", ordered_levels=["low_le_4", "intermediate_5_9", "high_ge_10"])
```

**`Compute` — multi-column rule (overwrite in place)**
```python
_derivations.Compute(
    name="edema_volume_cm3", sources=["perifocal_edema", "edema_volume_cm3"],
    fn=lambda d: d["edema_volume_cm3"].mask(
        d["perifocal_edema"].fillna(True).astype(float) == 0, 0),
    kind="continuous", overwrite=True,
    reason="No perifocal edema => edema volume is structurally 0, not missing.",
)
```

**Run the list**
```python
df, schema, derivation_log = _derivations.apply_derivations(
    df=df, schema=schema, derivations=DERIVATIONS,
    output_root=OUTPUT_ROOT, write_csv=True,
)
```

</details>


In [ ]:
_derivations = load("derivations")


def _ki67_midpoint(x):
    if pd.isna(x):
        return pd.NA
    parts = str(x).replace(",", ".").split("-")
    nums = [float(p) for p in parts]
    return sum(nums) / len(nums)
def _ki67_group(x):
    if pd.isna(x):
        return pd.NA
    if x <= 4:
        return "low_le_4"
    if x < 10:
        return "intermediate_5_9"
    return "high_ge_10"

DERIVATIONS = [
    _derivations.Apply(
        name="high_grade",
        source="who_grade",
        fn=lambda s: s.astype("Float64").pipe(lambda sf: (sf == 2) | (sf == 3)),
        kind="binary",
        active=True,
        overwrite=False,
        rule="who_grade in {2, 3}",
        reason="WHO 2021 CNS classification: grade 2/3 = high-grade meningioma.",
    ),
    _derivations.Apply(
        name="multiple_meningiomas",
        source="meningioma_count",
        fn=lambda s: s.astype("Float64") > 1,
        kind="binary",
        active=True,
        overwrite=False,
        rule="meningioma_count > 1",
        reason="",
    ),
    _derivations.Apply(
        name="ki67_mid",
        source="ki67_pct",
        fn=lambda s: s.map(_ki67_midpoint).astype("Float64"),
        kind="continuous",
        active=True,
        overwrite=False,
        rule="Midpoint of the reported Ki-67 range (e.g. \"5-10\" -> 7.5)",
        reason="",
    ),
    _derivations.Apply(
        name="ki67_group",
        source="ki67_mid",
        fn=lambda s: s.map(_ki67_group),
        kind="ordinal",
        ordered_levels=["low_le_4", "intermediate_5_9", "high_ge_10"],
        active=True,
        overwrite=False,
        rule="ki67_mid ≤ 4 / 5-9 / ≥ 10",
        reason="",
    ),
    _derivations.Compute(
        name="edema_volume_cm3",
        sources=["perifocal_edema", "edema_volume_cm3"],
        fn=lambda d: d["edema_volume_cm3"].mask(
            d["perifocal_edema"].fillna(True).astype(float) == 0, 0
        ),
        kind="continuous",
        active=True,
        overwrite=True,
        rule="edema_volume_cm3 set to 0 where perifocal_edema is absent",
        reason="Absent edema is a structural zero, not a missing measurement.",
    ),
    _derivations.Compute(
        name="edema_index",
        sources=["edema_volume_cm3", "tumor_volume"],
        fn=lambda d: d["edema_volume_cm3"] / d["tumor_volume"],
        kind="continuous",
        active=True,
        overwrite=False,
        rule="edema_volume_cm3 / tumor_volume",
        reason="Frati, Armocida et al., Tomography 2022;8(4):1987–1996, doi:10.3390/tomography8040166",
    ),
    _derivations.Apply(
        name="edema_index_ge1",
        source="edema_index",
        fn=lambda s: s.astype("Float64") >= 1,
        kind="binary",
        active=True,
        overwrite=False,
        rule="edema_index ≥ 1",
        reason="Frati, Armocida et al., Tomography 2022;8(4):1987–1996, doi:10.3390/tomography8040166",
    ),
    _derivations.Apply(
        name="edema_index_ge0.0617",
        source="edema_index",
        fn=lambda s: s.astype("Float64") >= 0.0617,
        kind="binary",
        active=True,
        overwrite=False,
        rule="edema_index ≥ 0.0617",
        reason="experimental threshold",
    ),
    _derivations.Apply(
        name="edema_volume_ge3.64",
        source="edema_volume_cm3",
        fn=lambda s: s.astype("Float64") >= 3.64,
        kind="binary",
        active=True,
        overwrite=False,
        rule="edema_volume_cm3 ≥ 3.64 cm³",
        reason="Adeli, Spille et al., Oncotarget 2018;9(89):35974-35982",
    ),
    _derivations.Apply(
        name="edema_volume_ge4.76",
        source="edema_volume_cm3",
        fn=lambda s: s.astype("Float64") >= 4.76,
        kind="binary",
        active=True,
        overwrite=False,
        rule="edema_volume_cm3 ≥ 4.76 cm³",
        reason="experimental threshold",
    ),
    _derivations.Apply(
        name="tumor_volume_ge13.95",
        source="tumor_volume",
        fn=lambda s: s.astype("Float64") >= 13.95,
        kind="binary",
        active=True,
        overwrite=False,
        rule="tumor_volume ≥ 13.95 cm³",
        reason="Shin, Kim, Cheong et al., PLoS One 2021;16(6):e0252945, n=205",
    ),
    _derivations.Apply(
        name="tumor_volume_ge15.1",
        source="tumor_volume",
        fn=lambda s: s.astype("Float64") >= 15.1,
        kind="binary",
        active=True,
        overwrite=False,
        rule="tumor_volume ≥ 15.1 cm³",
        reason="experimental threshold",
    ),
    _derivations.Apply(
        name="max_diameter_cm_gt6",
        source="max_diameter_cm",
        fn=lambda s: s.astype("Float64") > 6,
        kind="binary",
        active=True,
        overwrite=False,
        rule="max_diameter_cm > 6 cm",
        reason="Magill, Young, Chae et al. (UCSF), Neurosurg Focus 2018;44(4):E4, doi:10.3171/2018.1.FOCUS17752, n=1113",
    ),
    _derivations.Apply(
        name="max_diameter_cm_gt3",
        source="max_diameter_cm",
        fn=lambda s: s.astype("Float64") > 3,
        kind="binary",
        active=True,
        overwrite=False,
        rule="max_diameter_cm > 3 cm",
        reason="Magill, Young, Chae et al. (UCSF), Neurosurg Focus 2018;44(4):E4, doi:10.3171/2018.1.FOCUS17752, n=1113",
    ),
    _derivations.Apply(
        name="max_diameter_cm_ge3.81",
        source="max_diameter_cm",
        fn=lambda s: s.astype("Float64") >= 3.81,
        kind="binary",
        active=True,
        overwrite=False,
        rule="max_diameter_cm ≥ 3.81 cm",
        reason="experimental threshold",
    ),
    _derivations.Apply(
        name="adc_value_le0.72",
        source="adc_value",
        fn=lambda s: s.astype("Float64") <= 0.72,
        kind="binary",
        active=True,
        overwrite=False,
        rule="adc_value ≤ 0.72",
        reason="experimental threshold",
    ),

    #🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧 INACTIVE 🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧
    _derivations.BinNumeric(
        name="age_bins_10",
        source="age",
        bins=[-np.inf, 50, 60, 70, 80, np.inf],
        labels=["<50", "50-59", "60-69", "70-79", "80+"],
        kind="ordinal",
        active=False,
        overwrite=False,
        rule="10-year age bands: <50 / 50-59 / 60-69 / 70-79 / 80+",
        reason="Descriptive tables only; not a literature cutoff.",
    ),
    ]

# Derived parent->child map for MICE (see §08 notes): each derived column lists
# its source(s) so they can be recreated after imputation. edema_volume_cm3 is an
# in-place adjustment of an imputed variable, not a pure derived column, so it is
# intentionally excluded here.
DERIVED_DEPENDENCIES: dict[str, Sequence[str]] = {
    "age_bins": ["age"],
    "high_grade": ["who_grade"],            # analysis outcome (kept as predictor)
    "multiple_meningiomas": ["meningioma_count"],
    "ki67_mid": ["ki67_pct"],
    "ki67_group": ["ki67_mid"],
    "edema_volume_ge3.64": ["edema_volume_cm3"],
    "max_diameter_cm_gt6": ["max_diameter_cm"],
    "max_diameter_cm_gt3": ["max_diameter_cm"],
    "max_diameter_cm_ge3.81": ["max_diameter_cm"],
    "edema_index_ge1": ["edema_index"],
    "edema_index_ge0.0617": ["edema_index"],
    "edema_volume_ge4.76": ["edema_volume_cm3"],
    "tumor_volume_ge13.95": ["tumor_volume"],
    "tumor_volume_ge15.1": ["tumor_volume"],
    "adc_value_le0.72": ["adc_value"],
    }

In [ ]:
df, schema, derivation_log = _derivations.apply_derivations(
    df=df,
    schema=schema,
    derivations=DERIVATIONS,
    output_root=OUTPUT_ROOT,
    write_csv=True,
    )

#derivation_log

## 09 · Schema validation

✅ Fail loudly if the cleaned cohort drifts from the pandera contract you declare here.

<details>
<summary>🔧 How it works · <code>cleaning_phase/validation.py</code></summary>

Three helpers, two-step workflow:

- 📐 **`pandera_template(df)`** — prints a starter `DataFrameSchema` inferred from current dtypes: booleans → `nullable`, categories → `category_validation(...)` with live levels, numerics → `Float64` + wide bounds, datetimes → 2017→today, strings → `unique=True`. Copy the output and tighten it — the template is scaffolding, not gospel.
- 🏷️ **`category_validation(expected, ordered=False)`** — wraps the registered **`validate_category`** check (membership + optional level order). Serializable to JSON — no lambdas — so the schema can be exported and reused in the modelling notebook.
- ✅ **`pandera_validate(schema, df)`** — runs `schema_validation(df, lazy=True)` and collects *all* violations at once. Pass → saves `output/cleaning/schema_validation.json`. Fail → prints every failure case and stops.
- 🚨 `strict=True` on the schema → any extra or missing column fails.
- 🔁 The same `schema_validation` object is reused in §11 on **every** imputed draw, not just the first.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Step 1 — generate a starter schema from the current frame**
```python
pandera_template(df)
```

**Step 2 — refine + validate (tighten dtypes, nullability, ranges, ordered categories)**
```python
schema_validation = pa.DataFrameSchema(
    {
        "sex": pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("male", "female"))],
        ),
        "who_grade": pa.Column(
            dtype="category",
            nullable=False,
            checks=[*category_validation(("1", "2", "3"), ordered=True)],
        ),
        # ... one Column per kept column
    },
    strict=True,
)
pandera_validate(schema_validation, df)
```

**Validate every imputed draw (§11)**
```python
for _frame in imputed_frames:
    schema_validation(_frame, lazy=True)
```

</details>


In [ ]:
#🟧🟧🟧 print a starter schema from the current frame
#pandera_template(df)

In [ ]:
#🟧🟧🟧 copy-paste the output and tighten it
schema_validation = pa.DataFrameSchema(
    {
        'id': pa.Column(dtype=str, nullable=False, unique=True),
        'patient_code': pa.Column(dtype=str, unique=True),
        'entry_year': pa.Column(dtype="Int64"),
        'age': pa.Column(
            dtype="Float64", 
            nullable=True,
            checks=[
                pa.Check.ge(18),
                pa.Check.le(120)
            ]
        ),
        'sex': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("male", "female"))]
            ),
        
        #'histology_available': pa.Column(dtype="boolean",nullable=True,),

        'who_grade': pa.Column(
            dtype="category",
            nullable=False,
            checks=[*category_validation(("1", "2", "3"), ordered=True)]
            ),
        'progesterone_pos': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'ki67_pct': pa.Column(
            dtype=str,
            nullable=True
            ),
        'brain_invasion': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'hist_necrosis': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        
        'mri_date': pa.Column(
            dtype="datetime64",
            nullable=False,
            checks=[
                pa.Check.ge(pd.Timestamp("2017-01-01")),
                pa.Check.le(pd.Timestamp.today())
            ]),
        
        'side': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("right", "midline", "left"))]
            ),
        'tumor_location': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("non_skull_base", "skull_base"))]
            ),
        'meningioma_count': pa.Column(
            dtype="Int64",
            nullable=False
            ),
        'max_diameter_cm': pa.Column(
            dtype="Float64",
            nullable=True
            ),
        'tumor_volume': pa.Column(
            dtype="Float64",
            nullable=True
            ),
        
        #'additional_ct': pa.Column(dtype="boolean", nullable=True,),
        #'iv_contrast': pa.Column(dtype="boolean", nullable=True,),
        
        'tumor_episode': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("primary", "recurrent"), ordered=True)]
            ),
        'tumor_margin': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation({"regular", "irregular"})]
            ),
        'dural_tail': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'capsular_enhancement': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'heterogeneous_enhancement': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        
        'perifocal_edema': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'edema_volume_cm3': pa.Column(
            dtype="Float64",
            nullable=True
            ),
        
        'mass_effect': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'calcification': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'cystic_component': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'mri_necrosis': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'hemorrhage': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'hyperostosis': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'cortical_destruction': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        
        'dwi_hyperintensity': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        't2_hyperintensity': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        't1_hypointensity': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        
        'sinus_invasion': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("no_invasion", "sinus_invasion", "transsinus_extension"), ordered=True)]
            ),
        'transfalcine_extension': pa.Column(
            dtype="boolean",
            nullable=True,
            ),
        'adc_value': pa.Column(
            dtype="Float64",
            nullable=True
            ),
    

        #'age_bins_10': pa.Column(dtype="category", nullable=True, checks=[*category_validation(("<50", "50-59", "60-69", "70-79", "80+"), ordered=True)]),
        'high_grade': pa.Column(
            dtype="boolean",
            nullable=False
            ),
        'multiple_meningiomas': pa.Column(
            dtype="boolean",
            nullable=False
            ),
        'ki67_mid': pa.Column(
            dtype="Float64",
            nullable=True
            ),
        'ki67_group': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("low_le_4", "intermediate_5_9", "high_ge_10"), ordered=True)]
            ),
        'edema_index': pa.Column(
            dtype="Float64",
            nullable=True
            ),
        'edema_index_ge1': pa.Column(
            dtype="boolean",
            nullable=True
            ),
        'edema_index_ge0.0617': pa.Column(
            dtype="boolean",
            nullable=True
            ),
        'edema_volume_ge3.64': pa.Column(
            dtype="boolean",
            nullable=True
            ),
        'edema_volume_ge4.76': pa.Column(
            dtype="boolean",
            nullable=True
            ),
        'tumor_volume_ge13.95': pa.Column(
            dtype="boolean",
            nullable=True
            ),
        'tumor_volume_ge15.1': pa.Column(
            dtype="boolean",
            nullable=True
            ),
        'max_diameter_cm_gt6': pa.Column(
            dtype="boolean",
            nullable=True
            ),
        'max_diameter_cm_gt3': pa.Column(
            dtype="boolean",
            nullable=True
            ),
        'max_diameter_cm_ge3.81': pa.Column(
            dtype="boolean",
            nullable=True
            ),
        'adc_value_le0.72': pa.Column(
            dtype="boolean",
            nullable=True
            ),
    },
    strict=True
)
pandera_validate(schema_validation, df)

## 10 · DDA (pre-imputation)

📊 Descriptive pass on the typed + derived cohort, then an optional free-form peek before imputation.

<details>
<summary>🔧 How it works · univariate · <code>cleaning_phase/dda.py · run_dda</code></summary>

- 📐 Profiles overall shape, missing-cell %, and per-column summaries.
- 🖼️ Writes DDA tables and figures under `output/dda/`.
- 🛡️ Pure read-only checkpoint before imputation.
- ♻️ Does **not** change `df`.

</details>

<details>
<summary>🔧 How it works · bivariate · <code>cleaning_phase/dda.py · run_dda_bivariate</code></summary>

- 🟧 Edit `DDA_BIVARIATE = {x_col: [partner, …]}`. One SVG per pair.
- 🎨 Plot choice by types: scatter, overlapping KDEs, or grouped counts.
- 🧬 Partners may include derived columns (`age_bins_10`, `high_grade`, `ki67_group`, …).
- 🖼️ Files → `output/dda/figures_bivariate/` → report **2️⃣ DDA - bivariate**.
- ♻️ Does **not** change `df`.

</details>

<details>
<summary>🔧 How it works · trivariate · <code>cleaning_phase/dda.py · run_dda_trivariate</code></summary>

- 🟧 Edit `DDA_TRIVARIATE = {(x, y): [group, …]}`. One SVG per triple.
- 🎨 SciencePlots type matrix: cont×cont → scatter + **straight-line fit** / **smooth trend**; cont×cat → dodged box + strip; cat×cat → count facets. Ordered categoricals keep level order.
- 🎛️ `DDA_TRIVARIATE_STYLE` — pipeline default is `science`+`nature`+`no-latex` (shared via `plot_style`); swap commented variants if needed.
- 🏷️ Cont×cont legend mirrors the Bokeh reconnaissance style; corner badge shows total n (single-axis plots).
- 🖼️ Files → `output/dda/figures_trivariate/` → report **3️⃣ DDA - trivariate**.
- ♻️ Does **not** change `df`.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

```python
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

DDA_BIVARIATE = {
    "age": ["sex", "who_grade", "high_grade", "adc_value"],
    "high_grade": ["age_bins_10", "ki67_group", "adc_value"],
}
biv_paths = run_dda_bivariate(df, DDA_BIVARIATE, output_root=OUTPUT_ROOT)

DDA_TRIVARIATE = {
    ("max_diameter_cm", "tumor_volume"): ["high_grade", "sex"],
    ("tumor_volume", "tumor_location"): ["high_grade"],
    ("sex", "tumor_margin"): ["high_grade"],
}
tri_paths = run_dda_trivariate(
    df, DDA_TRIVARIATE, output_root=OUTPUT_ROOT,
    science_style=DDA_TRIVARIATE_STYLE,
)

# free-form peek
pd.crosstab(df["sex"], df["high_grade"], dropna=False)
```

</details>


In [ ]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)
#dda_tables["overall"]
#dda_tables["continuous"]
#dda_tables["categorical"]

### Bivariate by markers

🟧 Explicit pairs below — one seaborn SVG per pair.


In [ ]:
df.columns

In [ ]:
# 🟧 Explicit pairs: {x column: [partner columns, ...]} — one seaborn plot per pair
DDA_BIVARIATE = {
    "age": ['sex', 'brain_invasion',],
    "sex": ['brain_invasion', 'hist_necrosis', 'side', 'tumor_location', 'meningioma_count',
       'max_diameter_cm', 'tumor_volume', 'high_grade',
       'tumor_episode', 'perifocal_edema', 'edema_volume_cm3', 'mri_necrosis', 'adc_value', 'multiple_meningiomas',
       'edema_index', 'edema_index_ge1', 'edema_index_ge0.0617',
       'edema_volume_ge3.64', 'edema_volume_ge4.76', 'tumor_volume_ge13.95',
       'tumor_volume_ge15.1', 'max_diameter_cm_gt6', 'max_diameter_cm_gt3', 'max_diameter_cm_ge3.81',
       'adc_value_le0.72'
       ],
    "high_grade": ['sex', 'brain_invasion', 'hist_necrosis', 'side', 'tumor_location', 'meningioma_count',
       'max_diameter_cm', 'tumor_volume',
       'tumor_episode', 'tumor_margin', 'dural_tail', 'capsular_enhancement',
       'heterogeneous_enhancement', 'perifocal_edema', 'edema_volume_cm3',
       'mass_effect', 'calcification', 'cystic_component', 'mri_necrosis',
       'hemorrhage', 'hyperostosis', 'cortical_destruction',
       'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
       'sinus_invasion', 'transfalcine_extension', 'adc_value', 'multiple_meningiomas',
       'edema_volume_ge3.64', 'edema_volume_ge4.76', 'tumor_volume_ge13.95',
       'tumor_volume_ge15.1', 'max_diameter_cm_gt6', 'max_diameter_cm_gt3', 'max_diameter_cm_ge3.81',
       'adc_value_le0.72'
       ],
    "adc_value": ['sex', 'brain_invasion', 'hist_necrosis', 'side', 'tumor_location', 'meningioma_count',
       'max_diameter_cm', 'tumor_volume',
       'tumor_episode', 'tumor_margin', 'dural_tail', 'capsular_enhancement',
       'heterogeneous_enhancement', 'perifocal_edema', 'edema_volume_cm3',
       'mass_effect', 'calcification', 'cystic_component', 'mri_necrosis',
       'hemorrhage', 'hyperostosis', 'cortical_destruction',
       'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
       'sinus_invasion', 'transfalcine_extension', 'adc_value', 'multiple_meningiomas',
       'edema_volume_ge3.64', 'edema_volume_ge4.76', 'tumor_volume_ge13.95',
       'tumor_volume_ge15.1', 'max_diameter_cm_gt6', 'max_diameter_cm_gt3', 'max_diameter_cm_ge3.81',
       'adc_value_le0.72'
       ],
}

biv_paths = run_dda_bivariate(df, DDA_BIVARIATE, output_root=OUTPUT_ROOT)
#len(biv_paths)
#biv_paths

### Ultimate trivariate by markers

🟧 Explicit triples below — pairs compared across groups via **SciencePlots** (cont/cat × cont/cat; ordered levels respected). Cont×cont legends use `straight-line fit` / `smooth trend`.


In [ ]:
# 🟧 Explicit triples: {(x, y): [group columns, ...]} — one SciencePlots figure per triple
DDA_TRIVARIATE = {
    #("max_diameter_cm_gt6", "tumor_volume_ge13.95"): ["high_grade", "sex"],
    #("tumor_volume_ge13.95", "tumor_location"): ["high_grade"],
    #("sex", "tumor_margin"): ["high_grade"],
}

#DDA_TRIVARIATE_STYLE = ["science", "no-latex"]
DDA_TRIVARIATE_STYLE = ["science", "nature", "no-latex"]
# DDA_TRIVARIATE_STYLE = ["science", "ieee", "no-latex"]
# DDA_TRIVARIATE_STYLE = ["science", "bright", "grid", "no-latex"]

tri_paths = run_dda_trivariate(
    df, DDA_TRIVARIATE, output_root=OUTPUT_ROOT, science_style=DDA_TRIVARIATE_STYLE,
)
#len(tri_paths)
#tri_paths

### Free-form peek

🔬 Optional scratch space — last look before imputation. Not part of the pipeline contract.


## 11 · Imputation (MICE or simple)

🧬 Fill the gaps — generate **m** completed datasets for Rubin's-rules pooling downstream.

<details>
<summary>🔧 How it works · <code>cleaning_phase/missingness_resolution.py</code></summary>

Three engines, pick one:

- 🥇 **`proper_mice_impute`** (primary) — formal mixed-type MICE via R's `mice` (`heavy_machinery/scripts/run_mice.R`) (needs R + `mice`/`jsonlite`). Drops non-outcome derived columns, imputes their sources, recreates them from `derivations=DERIVATIONS`, and saves diagnostics under `output/missingness/mice/`.
- 🧪 **`rf_chained_impute`** (optional) — RandomForest chained imputation; sensitivity check only, **not** valid for Rubin pooling.
- 🩹 **`simple_impute_stage`** (fallback) — median/mode, single dataset.

Tips:

- 🔢 `m≥10` (e.g. 20) for publication; `m=3` for a quick smoke run.
- 💾 Writes `unimputed_df.parquet` + `mice_imputed_df.parquet`, then pandera-validates every draw.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Primary — formal MICE (R)**
```python
imputed_frames = proper_mice_impute(
    df, schema,
    m=3, max_iter=5,             # 🔁 3/5 smoke · 20/20 publication
    random_state=42,
    analysis_outcome="high_grade",
    derived_dependencies=DERIVED_DEPENDENCIES,
    derivations=DERIVATIONS,
    output_root=OUTPUT_ROOT,
)
for _frame in imputed_frames:
    schema_validation(_frame, lazy=True)
```

**Optional — RF chained (sensitivity only)**
```python
imputed_frames_rf = rf_chained_impute(
    df, schema, m=3, max_iter=10, n_estimators=20, output_root=OUTPUT_ROOT,
)
```

**Fallback — simple median/mode**
```python
imputed_frames = [simple_impute_stage(df, schema, OUTPUT_ROOT, impute_binary=False)]
```

</details>


In [ ]:
#🟧🟧🟧 Formal mixed-type MICE (R mice) — primary imputation (see §11 notes)
# Requires R + packages: install.packages(c("mice", "jsonlite"))

imputed_frames = proper_mice_impute(
    df,
    schema,
    m=20,                     # 3 smoke | 20 publication
    max_iter=20,              # 5 smoke | 20 publication
    random_state=42,
    analysis_outcome="high_grade",
    derived_dependencies=DERIVED_DEPENDENCIES,
    derivations=DERIVATIONS,
    output_root=OUTPUT_ROOT,
)

# Pandera validation on EVERY completed dataset (not only draw 1).
for _i, _frame in enumerate(imputed_frames, start=1):
    schema_validation(_frame, lazy=True)
    print(f"✅ Pandera validated imputed frame {_i}/{len(imputed_frames)}")

In [ ]:
#🟧🟧🟧 OPTIONAL sensitivity analysis — RF chained imputation (NOT formal MICE)
# Post-hoc Bernoulli sampling; Rubin pooling is NOT supported on these draws.
# Manifest marks proper_multiple_imputation=False. Use only as a sensitivity check.

#imputed_frames_rf = rf_chained_impute(
#    df, schema, m=3, max_iter=10, n_estimators=20, output_root=OUTPUT_ROOT,
#)


#🟧🟧🟧 Skip MICE — median/mode imputation (binary left NaN by default)

#imputed_frames = [simple_impute_stage(df, schema, OUTPUT_ROOT, impute_binary=False)]

#display(imputation_audit(
#    load_unimputed_dataset(OUTPUT_ROOT),
#    load_modeling_frames(OUTPUT_ROOT)[0],
#    schema,
#    INFERENTIAL_MANUAL_PREDICTORS,
#    impute_binary=False,
#))

#print("NaN count (all columns):", load_modeling_frames(OUTPUT_ROOT)[0].isna().sum().sum())

## 12 · Save handoff datasets

📦 Reload and validate the parquets so the modelling notebook can trust them.

<details>
<summary>🔧 How it works · <code>cleaning_phase/dataset_handoff.py</code></summary>

- 💾 Parquets are written during imputation (§11); this step doesn't re-impute.
- 📋 Re-exports `schema/schema_summary.csv` from the final in-memory schema.
- 🔁 `validate_handoff_datasets` reloads every parquet and round-trip validates it (dtypes + row counts).
- 🤝 Guarantees `meningioma-modelling.ipynb` reads exactly what was saved.
- 🏷️ Set `imputation_method` to match what you ran (`"mice"` vs `"simple"`).

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**MICE handoff (multiple draws)**
```python
imputation_method = "mice"
validate_handoff_datasets(
    OUTPUT_ROOT, df_unimputed=df, schema=schema,
    imputation_method="mice", imputed_frames=imputed_frames,
)
```

**Simple handoff (single frame)**
```python
imputation_method = "simple"
validate_handoff_datasets(
    OUTPUT_ROOT, df_unimputed=df, schema=schema,
    imputation_method="simple", imputed_single=imputed_frames[0],
)
```

</details>


In [ ]:
# Set imputation_method to "simple" if you used simple_impute_stage above instead of MICE.
imputation_method = "mice"

if imputation_method == "mice":
    validate_handoff_datasets(
        OUTPUT_ROOT,
        df_unimputed=df,
        schema=schema,
        imputation_method="mice",
        imputed_frames=imputed_frames,
        )
else:
    imputed_single = imputed_frames[0]
    validate_handoff_datasets(
        OUTPUT_ROOT,
        df_unimputed=df,
        schema=schema,
        imputation_method="simple",
        imputed_single=imputed_single,
        )